# Hafta 12 - Evrişimli Sinir Ağları (CNN) Temelleri

Bu defterde CIFAR-10 veri seti üzerinde CNN mimarisini öğreneceğiz.

## İçerik
1. CNN Mimarisi Nedir?
2. Filtre/Çekirdek (Kernel) Kavramı
3. CIFAR-10 Veri Setini Yükleme
4. Veri Görselleştirme ve Ön İşleme
5. CNN Modeli Oluşturma
6. Eğitim ve Değerlendirme
7. Veri Artırma (Data Augmentation)

## 1. CNN Mimarisi Nedir?

**Evrişimli Sinir Ağları (Convolutional Neural Networks - CNN)**, özellikle görüntü tanıma ve sınıflandırma görevlerinde kullanılan derin öğrenme mimarisidir.

### Temel Katmanlar

| Katman | Açıklama |
|--------|----------|
| **Conv2D** | Evrişim katmanı — görüntü üzerinde filtreler kaydırarak özellik haritaları (feature maps) oluşturur |
| **MaxPooling2D** | Havuzlama katmanı — özellik haritalarını küçülterek hesaplama maliyetini azaltır ve önemli özellikleri korur |
| **Flatten** | Düzleştirme katmanı — çok boyutlu veriyi tek boyutlu vektöre dönüştürür |
| **Dense** | Tam bağlı katman — sınıflandırma için kullanılır |

### CNN Akış Şeması

```
Girdi Görüntü (32x32x3)
    ↓
Conv2D (32 filtre, 3x3) → ReLU
    ↓
MaxPooling2D (2x2)
    ↓
Conv2D (64 filtre, 3x3) → ReLU
    ↓
MaxPooling2D (2x2)
    ↓
Flatten
    ↓
Dense (64, ReLU)
    ↓
Dense (10, Softmax) → Sınıf Olasılıkları
```

## 2. Filtre/Çekirdek (Kernel) Kavramı

Filtre (kernel), görüntü üzerinde kayan küçük bir matristir. Her filtre belirli bir özelliği (kenar, köşe, doku vb.) algılar.

### Evrişim İşlemi Görselleştirmesi

```
Girdi Görüntü (5x5):          Filtre (3x3):         Çıktı (3x3):
┌───┬───┬───┬───┬───┐        ┌───┬───┬───┐        ┌───┬───┬───┐
│ 1 │ 0 │ 1 │ 0 │ 1 │        │ 1 │ 0 │ 1 │        │ 4 │ 3 │ 4 │
├───┼───┼───┼───┼───┤        ├───┼───┼───┤        ├───┼───┼───┤
│ 0 │ 1 │ 0 │ 1 │ 0 │   *    │ 0 │ 1 │ 0 │   =    │ 2 │ 4 │ 2 │
├───┼───┼───┼───┼───┤        ├───┼───┼───┤        ├───┼───┼───┤
│ 1 │ 0 │ 1 │ 0 │ 1 │        │ 1 │ 0 │ 1 │        │ 4 │ 3 │ 4 │
├───┼───┼───┼───┼───┤        └───┴───┴───┘        └───┴───┴───┘
│ 0 │ 1 │ 0 │ 1 │ 0 │
├───┼───┼───┼───┼───┤
│ 1 │ 0 │ 1 │ 0 │ 1 │
└───┴───┴───┴───┴───┘
```

**Filtre türleri:**
- **Kenar algılama filtresi**: Yatay/dikey kenarları bulur
- **Bulanıklaştırma filtresi**: Görüntüyü yumuşatır
- **Keskinleştirme filtresi**: Detayları belirginleştirir

CNN eğitilirken bu filtreler **otomatik olarak öğrenilir** — elle tanımlamaya gerek yoktur!

## 3. CIFAR-10 Veri Setini Yükleme

### Kütüphanelerin Yüklenmesi

Projede kullanacağımız kütüphaneleri içe aktarıyoruz:

| Kütüphane | Amacı |
|-----------|-------|
| `matplotlib` | Grafik ve görselleştirme |
| `numpy` | Sayısal hesaplamalar ve dizi işlemleri |
| `tensorflow` | Derin öğrenme modelleri oluşturma ve eğitme |
| `warnings` | Uyarı mesajlarını yönetme |


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow sürümü: {tf.__version__}")

### Veri Setinin Yüklenmesi

Aşağıdaki kodda veri setini yüklüyoruz ve temel bilgilerine (boyut, sütunlar, ilk satırlar) bakıyoruz. Bu adım her veri bilimi projesinin başlangıcıdır.

In [ ]:
# CIFAR-10 veri setini yükle
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Sınıf isimleri (Türkçe)
sinif_isimleri = ['Uçak', 'Araba', 'Kuş', 'Kedi', 'Geyik', 
                  'Köpek', 'Kurbağa', 'At', 'Gemi', 'Kamyon']

print(f"Eğitim seti boyutu: {X_train.shape}")
print(f"Test seti boyutu: {X_test.shape}")
print(f"Eğitim etiketi boyutu: {y_train.shape}")
print(f"Görüntü boyutu: {X_train.shape[1]}x{X_train.shape[2]}x{X_train.shape[3]}")
print(f"Sınıf sayısı: {len(sinif_isimleri)}")
print(f"Piksel değer aralığı: [{X_train.min()}, {X_train.max()}]")

## 4. Veri Görselleştirme ve Ön İşleme

### Çoklu Grafik Paneli

Birden fazla grafiği yan yana veya alt alta çizdirerek karşılaştırmalı analiz yapıyoruz.

In [ ]:
# Örnek görüntüleri görselleştir
plt.figure(figsize=(15, 6))
for i in range(20):
    plt.subplot(2, 10, i + 1)
    plt.imshow(X_train[i])
    plt.title(sinif_isimleri[y_train[i][0]], fontsize=9)
    plt.axis('off')

plt.suptitle('CIFAR-10 Örnek Görüntüler', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Çoklu Grafik Paneli

Birden fazla grafiği yan yana veya alt alta çizdirerek karşılaştırmalı analiz yapıyoruz.

In [ ]:
# Her sınıftan birer örnek göster
plt.figure(figsize=(15, 3))
for i, sinif in enumerate(sinif_isimleri):
    idx = np.where(y_train.flatten() == i)[0][0]
    plt.subplot(1, 10, i + 1)
    plt.imshow(X_train[idx])
    plt.title(sinif, fontsize=10)
    plt.axis('off')

plt.suptitle('Her Sınıftan Bir Örnek', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Görselleştirme

Aşağıdaki grafikle veriyi görsel olarak inceliyoruz. Görselleştirme, sayısal analizlerin ötesinde örüntüleri ve anormallikleri fark etmemizi sağlar.

In [ ]:
# Sınıf dağılımı
sinif_sayilari = np.bincount(y_train.flatten())

plt.figure(figsize=(10, 5))
plt.bar(sinif_isimleri, sinif_sayilari, color='steelblue', edgecolor='black')
plt.title('Eğitim Setindeki Sınıf Dağılımı', fontsize=14, fontweight='bold')
plt.xlabel('Sınıf')
plt.ylabel('Örnek Sayısı')
plt.xticks(rotation=45)
for i, v in enumerate(sinif_sayilari):
    plt.text(i, v + 50, str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

### Streamlit Uygulaması

Streamlit ile interaktif bir veri uygulaması oluşturuyoruz. Streamlit, Python kodunu otomatik olarak web arayüzüne dönüştürür.

In [ ]:
# Normalizasyon: 0-255 aralığını 0-1 aralığına dönüştür
X_train_norm = X_train.astype('float32') / 255.0
X_test_norm = X_test.astype('float32') / 255.0

print(f"Normalizasyon sonrası piksel aralığı: [{X_train_norm.min():.1f}, {X_train_norm.max():.1f}]")
print(f"Eğitim seti şekli: {X_train_norm.shape}")
print(f"Test seti şekli: {X_test_norm.shape}")

## 5. CNN Modeli Oluşturma

### Sinir Ağı Modeli Oluşturma

Keras Sequential API ile katman katman sinir ağı modeli inşa ediyoruz. Her katmanın kendine özgü bir görevi vardır (özellik çıkarma, boyut düşürme, sınıflandırma).

In [ ]:
# CNN modeli oluştur
model = tf.keras.Sequential([
    # İlk evrişim bloğu
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    
    # İkinci evrişim bloğu
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    
    # Sınıflandırma katmanları
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

model.summary()

### Modeli derle

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Modeli derle
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Model başarıyla derlendi!")
print(f"Kayıp fonksiyonu: sparse_categorical_crossentropy")
print(f"Optimizasyon: Adam")
print(f"Metrik: accuracy")

## 6. Eğitim ve Değerlendirme

### Model Eğitimi

Aşağıdaki kodda modeli eğitim verisi üzerinde eğitiyoruz (`.fit()`). Eğitim sonrası test verisi üzerinde tahmin yapıp (`.predict()`) başarı metriklerini hesaplıyoruz.

In [ ]:
# Modeli eğit
gecmis = model.fit(
    X_train_norm, y_train,
    epochs=15,
    batch_size=64,
    validation_data=(X_test_norm, y_test),
    verbose=1
)

### Dağılım Görselleştirmesi

Aşağıdaki grafikte verinin dağılımını histogram ile inceliyoruz. Dağılımın şekli (normal, çarpık, bimodal) hangi istatistiksel yöntemlerin uygulanabileceğini belirler.

In [ ]:
# Eğitim grafiklerini çiz
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Doğruluk grafiği
ax1.plot(gecmis.history['accuracy'], label='Eğitim Doğruluğu', linewidth=2)
ax1.plot(gecmis.history['val_accuracy'], label='Doğrulama Doğruluğu', linewidth=2)
ax1.set_title('Model Doğruluğu', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Doğruluk')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Kayıp grafiği
ax2.plot(gecmis.history['loss'], label='Eğitim Kaybı', linewidth=2)
ax2.plot(gecmis.history['val_loss'], label='Doğrulama Kaybı', linewidth=2)
ax2.set_title('Model Kaybı', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Kayıp')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Test seti üzerinde değerlendir

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Test seti üzerinde değerlendir
test_kaybi, test_dogrulugu = model.evaluate(X_test_norm, y_test, verbose=0)
print(f"Test Kaybı: {test_kaybi:.4f}")
print(f"Test Doğruluğu: {test_dogrulugu:.4f} ({test_dogrulugu*100:.2f}%)")

### Çoklu Grafik Paneli

Birden fazla grafiği yan yana veya alt alta çizdirerek karşılaştırmalı analiz yapıyoruz.

In [ ]:
# Tahminleri görselleştir
tahminler = model.predict(X_test_norm[:25])

plt.figure(figsize=(15, 10))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.imshow(X_test[i])
    
    tahmin_sinif = np.argmax(tahminler[i])
    gercek_sinif = y_test[i][0]
    guvence = tahminler[i][tahmin_sinif] * 100
    
    renk = 'green' if tahmin_sinif == gercek_sinif else 'red'
    plt.title(f"T: {sinif_isimleri[tahmin_sinif]}\nG: {sinif_isimleri[gercek_sinif]}\n%{guvence:.0f}",
              fontsize=8, color=renk)
    plt.axis('off')

plt.suptitle('Tahmin Sonuçları (T: Tahmin, G: Gerçek)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Korelasyon Analizi ve Isı Haritası

Özellikler arasındaki ilişkiyi korelasyon matrisi ve ısı haritası ile görselleştiriyoruz. Yüksek korelasyonlu özellikler modelin en çok yararlanacağı özelliklerdir.

In [ ]:
# Karışıklık matrisi
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

tum_tahminler = np.argmax(model.predict(X_test_norm), axis=1)
cm = confusion_matrix(y_test.flatten(), tum_tahminler)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=sinif_isimleri, yticklabels=sinif_isimleri)
plt.title('Karışıklık Matrisi', fontsize=14, fontweight='bold')
plt.xlabel('Tahmin Edilen')
plt.ylabel('Gerçek')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7. Veri Artırma (Data Augmentation)

Veri artırma, mevcut eğitim verilerini döndürerek, kaydırarak, aynalayarak vb. yeni örnekler oluşturma tekniğidir. Bu, modelin daha iyi genelleştirmesine yardımcı olur.

### Neden Veri Artırma?
- Aşırı öğrenmeyi (overfitting) azaltır
- Eğitim veri çeşitliliğini artırır
- Model performansını iyileştirir

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Veri artırma jeneratörü oluştur
veri_artirma = ImageDataGenerator(
    rotation_range=15,           # Rastgele döndürme (±15 derece)
    width_shift_range=0.1,       # Yatay kaydırma
    height_shift_range=0.1,      # Dikey kaydırma
    horizontal_flip=True,        # Yatay aynalama
    zoom_range=0.1               # Yakınlaştırma/uzaklaştırma
)

# Bir görüntünün artırılmış versiyonlarını göster
ornek_goruntu = X_train_norm[0:1]  # İlk görüntüyü al

plt.figure(figsize=(15, 3))
plt.subplot(1, 8, 1)
plt.imshow(ornek_goruntu[0])
plt.title('Orijinal', fontsize=9)
plt.axis('off')

artirma_iter = veri_artirma.flow(ornek_goruntu, batch_size=1)
for i in range(7):
    artirilmis = artirma_iter.next()
    plt.subplot(1, 8, i + 2)
    plt.imshow(artirilmis[0])
    plt.title(f'Artırılmış {i+1}', fontsize=9)
    plt.axis('off')

plt.suptitle('Veri Artırma Örnekleri', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nVeri artırma, modelin farklı açı ve konumlardaki nesneleri tanımasına yardımcı olur.")

## Özet

Bu defterde öğrendiklerimiz:

1. **CNN Mimarisi**: Conv2D, MaxPooling2D, Flatten, Dense katmanları
2. **Filtre/Çekirdek**: Görüntüdeki özellikleri algılayan öğrenilebilir matrisler
3. **CIFAR-10**: 10 sınıflı 60.000 renkli görüntü veri seti
4. **Model Eğitimi**: Derleme, eğitim, doğruluk/kayıp grafikleri
5. **Veri Artırma**: Eğitim veri çeşitliliğini artırma teknikleri

### Sonraki Adımlar
- Transfer Learning ile daha güçlü modeller
- Daha derin CNN mimarileri (VGG, ResNet)
- Gerçek dünya uygulamaları